# ✈️ Airline Passenger Satisfaction Prediction using Logistic Regression
### Built by: Data Scientist & Customer Experience Strategist

## 1. Project Overview & Exploratory Data Analysis
This analysis leverages a large-scale passenger survey dataset to construct a predictive classification model using **Scikit-learn's Binomial Logistic Regression**. The core goal is to determine the primary elements driving customer delight, evaluate classification trade-offs using a robust matrix framework, and deliver data-backed strategic insights to optimize airline passenger retention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, accuracy_score, roc_curve, auc

# 1. Load the airline passenger satisfaction dataset
df = pd.read_csv('617ec7a0-b7f1-423e-b810-23f59803ffb6.csv')

# 2. Inspect missing records and dimension sizes
print('Initial Dataset Shape:', df.shape)
print('\n--- Missing Values Per Column ---')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Prune missing rows within the arrival delay feature to preserve signal stability
df = df.dropna()
print('\nCleaned Dataset Shape:', df.shape)

# 3. Check baseline target classification distribution 
print('\n--- Target Label (satisfaction) Proportions ---')
print(df['satisfaction'].value_counts(normalize=True))

### Exploratory Visualization
We inspect how different passenger travel tiers associate visually with raw passenger satisfaction margins before modeling.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Class', hue='satisfaction', palette='Set2')
plt.title('Passenger Satisfaction Distribution Across Travel Classes')
plt.xlabel('Flight Ticket Class')
plt.ylabel('Total Passenger Count')
plt.legend(title='Satisfaction Status')
plt.show()

## 2. Feature Engineering & One-Hot Encoding
To render the categorical features compatible with Scikit-learn algorithms, we convert our target indicator (`satisfied` = 1, `dissatisfied` = 0) and execute full binary dummy-encoding on multi-tiered variables (`Customer Type`, `Type of Travel`, `Class`) utilizing `drop_first=True` to preserve independent variable boundaries.

In [ ]:
# Map categorical satisfaction classes into binary integers
df['satisfaction_encoded'] = (df['satisfaction'] == 'satisfied').astype(int)

# Split out target label from features 
X_raw = df.drop(columns=['satisfaction', 'satisfaction_encoded'])
y = df['satisfaction_encoded']

# Implement full One-Hot Encoding on all multi-categorical features
X_encoded = pd.get_dummies(X_raw, columns=['Customer Type', 'Type of Travel', 'Class'], drop_first=True, dtype=float)

print('Encoded Feature Space Matrix Shape:', X_encoded.shape)
print('\n--- Feature Schema Post-Encoding ---')
print(X_encoded.columns.tolist())

## 3. Stratified Train-Test Split & Feature Normalization
We slice the encoded matrices into an **80/20 train-test split**. Using a stratified parameter maintains perfect label distributions between subsets. We then execute a `StandardScaler` transformation over the data space so all model coefficients can be compared directly as clear metrics of feature importance.

In [ ]:
# Segment training and testing sets with target stratification
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Apply standard normalization transformations
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print('Training Inputs Matrix Shape:', X_train.shape)
print('Testing Inputs Matrix Shape :', X_test.shape)

## 4. Logistic Regression Classifier Implementation
We fit a binomial Logistic Regression estimator to our scaled matrix space, utilizing an expanded iteration ceiling to ensure stable mathematical convergence.

In [ ]:
# Initialize and fit our Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Extract deterministic classification decisions and continuous probabilities
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

print('Logistic Regression estimation phase completed successfully!')

## 5. Model Evaluation & Error Metrics
To strictly evaluate true positive vs. false positive classification trade-offs, we output the complete accuracy matrix, a precision/recall framework, and the Area Under the ROC Curve (AUC).

In [ ]:
# Generate primary evaluation parameters
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print('--- Core Performance Vector Summary ---')
print(f'Model Accuracy Score : {accuracy:.4f}')
print(f'Model Precision Score: {precision:.4f}')
print(f'Model Recall Score   : {recall:.4f}')

print('\n--- Comprehensive Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['Dissatisfied', 'Satisfied']))

# Generate the absolute Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Generate a professional heatmap visualization
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Dissatisfied', 'Predicted Satisfied'],
            yticklabels=['Actual Dissatisfied', 'Actual Satisfied'])
plt.title('Model Confusion Matrix Matrix Heatmap')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.show()

### Receiver Operating Characteristic (ROC) Curve Exploration

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.legend(loc='lower right')
plt.show()

## 6. Coefficient Interpretation & Passenger Drivers Analysis
We isolate and sort the standardized feature weights from our trained classification engine to uncover the strongest positive and negative operational drivers of passenger sentiment.

In [ ]:
coefficients = log_reg.coef_[0]
features_list = X_encoded.columns

coef_df = pd.DataFrame({
    'Feature': features_list,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Standardized Logistic Regression Coefficients (Feature Importance)')
plt.xlabel('Coefficient Weight (Impact on Log-Odds of Satisfaction)')
plt.ylabel('Feature Attributes')
plt.show()

print('--- All Features Sorted by Coefficient Structural Weight ---')
print(coef_df.to_string(index=False))

## 7. Strategic Business Insights & Recommendations

### Final Portable Classification Mathematical Model
The baseline probability of an individual passenger reporting a **Satisfied** travel status is computed via the following formula structure:

$$P(\text{Satisfied}) = \frac{1}{1 + e^{-z}}$$

$$z = \beta_0 + \beta_1(\text{Age}) + \beta_2(\text{Flight Distance}) + \dots + \beta_n(\text{Class\_Eco})$$

### Performance Metric Evaluation
* **Accuracy (82.53%):** The classification model maps overall outcomes reliably for over 82% of all validation records.
* **Precision (83.92%):** When our model marks an experience as 'Satisfied', it is correct **83.92% of the time**, providing confidence for capital investments.
* **Recall / Sensitivity (84.21%):** The model captures **84.21% of all truly satisfied passengers**, indicating high structural sensitivity across our feature matrix.
* **AUC-ROC Score (0.9026):** A score of **0.903** indicates exceptional model discrimination capability.

### Core Passenger Experience Discoveries
1. **The Primary Driver—Inflight Entertainment (+0.9730):** Holding all other factors constant, an increase in the standardized rating for inflight entertainment yields the single largest boost to passenger satisfaction logs. 
2. **Foundational Amenities:** **Seat Comfort (+0.4096)**, **On-board Service (+0.3990)**, and **Check-in Service (+0.3584)** represent crucial cornerstones of passenger sentiment.
3. **Key Satisfaction Risks:** Economy flyers (`Class_Eco`: -0.3599) and casual leisure travelers (`Type of Travel_Personal Travel`: -0.3507) exhibit a significantly lower baseline satisfaction probability, highlighting clear areas of operational friction.

### Actionable Strategic Executive Steps
* **Optimize Media Content Spend:** Allocate capital budgets heavily toward digital entertainment systems and media asset updates. Because `Inflight entertainment` exhibits more than double the positive weight of secondary features, it serves as your highest-leverage asset for elevating customer delight.
* **Mitigate Economy Class Friction:** Design proactively targeted operational buffers for Economy Class segments, such as self-service digital check-in alerts or automated priority bag queues, to counteract the predictable baseline drop in satisfaction probability (-0.3599).
* **Refine Terminal Readiness:** Conduct specialized performance reviews for gate, on-board, and desk personnel to stabilize customer touchpoints across front-line operations.